**BERN02 Exercise: Regression**

Name: Yang Shann Wen

Date: 1 September 2026

In [2]:
import pandas as pd
import numpy as np

In [10]:
# Import and load dataset
df = pd.read_csv('/pollution_cleaneddata.csv')

In [11]:
# Extract the vector of observations of the predictor: % of families with income < $3000 (POOR)
x_data = df['POOR']

# Extract the vector of observations of the response variable: Total age-adjusted mortality rate per 100,000 (MORT)
y_data = df['MORT']

# Convert into numpy array for calculations
x_data = np.array(x_data)
y_data = np.array(y_data)

In [21]:
# Function for performing predictions with Local Regression
def loess(y, x, k, x0):
  """
  Parameters:
  y: The response variable observations
  x: The predictor observations
  k: The number of nearest neighboring points to include in each local regression
  x0: The target values to be predicted

  Outputs:
  pred: The list of predicted expected values
  se: The list of standard deviations of the expected values

  """

  # Empty list to store my predicted expected values and standard errors
  pred = []
  se = []

  # Run the function for every target value in x0
  for i in x0:

    # Calculate distances between every data point x and the target value
    distances = np.abs(x-i)

    # Pick the k nearest points
    nearest_index = np.argsort(distances)[:k]

    # Fit an unweighted OLS line for the local area
    # Extract the x and y values for these k nearest points
    x_neighbors = x[nearest_index]
    y_neighbors = y[nearest_index]

    # Calculate the average of these k nearest points
    x_mean = np.mean(x_neighbors)
    y_mean = np.mean(y_neighbors)

    # Calculate the slope (beta_1): simple regression formula
    numerator = np.sum((x_neighbors-x_mean) * (y_neighbors-y_mean))
    denominator = np.sum((x_neighbors - x_mean)**2)
    beta_1 = numerator / denominator

    # Calculate the intercept (beta_0)
    beta_0 = y_mean - (beta_1 * x_mean)

    # Plug in target value to make predictions
    y_predicted = beta_0 + (beta_1 * i)
    pred.append(float(y_predicted))

    # Calculate the variance of the error (sigma^2)
    e = y_neighbors - beta_0 - (beta_1 * x_neighbors)
    variance = (np.sum(e**2)) / (k-2)

    # Calculate the variance of the expected value
    h = ((i - x_mean)**2) / denominator
    variance_expected_value = variance * ((1/k) + h)

    # Square root for standard deviation
    standard_error = np.sqrt(variance_expected_value)
    se.append(float(standard_error))

  return pred, se

In [36]:
# Set parameters for prediction
k = 10
x0 = [10,18,25]

# Run the function
pred, se = loess(y_data, x_data, k, x0)

print("--- Local Regression Predictions ---")
print("POOR (%):       ", x0)
print("MORT:           ", pred)
print("Standard Errors:", se)

--- Local Regression Predictions ---
POOR (%):        [10, 18, 25]
MORT:            [901.8106176577891, 957.2334338613927, 1012.1632569203205]
Standard Errors: [20.935046418177652, 16.545832879992183, 27.593430462043425]


In [37]:
# The target poverty levels you need to predict
x0 = [10, 18, 25]

# A list of different neighborhood sizes to test
k_values = [5, 10, 15]

print("--- Comparing Different k Values ---")

# Loop through each k value and print a mini-report
for k in k_values:
    # Run your custom function
    predictions, standard_errors = loess(y_data, x_data, k, x0)

    # Rounding to 2 decimal places so it is easier to read!
    predictions_rounded = np.round(predictions, 2)
    se_rounded = np.round(standard_errors, 2)

    print(f"\nResults when k = {k}:")
    print(f"  Predicted Mortality: {predictions_rounded}")
    print(f"  Standard Errors:     {se_rounded}")

--- Comparing Different k Values ---

Results when k = 5:
  Predicted Mortality: [ 870.83  971.5  1024.88]
  Standard Errors:     [21.61 24.43 28.62]

Results when k = 10:
  Predicted Mortality: [ 901.81  957.23 1012.16]
  Standard Errors:     [20.94 16.55 27.59]

Results when k = 15:
  Predicted Mortality: [ 902.91  959.19 1009.03]
  Standard Errors:     [18.08 20.65 25.49]
